<a href="https://colab.research.google.com/github/ShonenMind/derm-research/blob/master/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Benchmarking Open-Source AI Models for Dermatological Diagnosis

**BCN20000 Dermoscopy Dataset · Google Colab Notebook**

Authors: Rohan Rattan, Nawoda Wijesooriya, Arjun Damerla
Affiliation: Dept. of Dermatology, UC San Diego

## Section 0 — How to use this notebook

### What this notebook does

This notebook trains and evaluates 6 open-source AI models on the task of classifying dermoscopic skin lesion images from the BCN20000 dataset into 8 diagnostic categories. The goal is to identify which model architecture performs best — and at which specific diagnoses — to inform the development of reliable AI-assisted dermatology tools.

### Before you start — checklist

- [ ] Set your runtime to GPU: Runtime → Change runtime type → GPU (T4 is free; A100 via Colab Pro)
- [ ] Mount your Google Drive (Cell 1.2)
- [ ] Run Cell 2.0 (it builds `train.csv` / `val.csv` / `test.csv` automatically from `metadata.csv`)
- [ ] Update `DATA_SOURCE_ROOT` in Cell 1.5 to point at your BCN20000 folder
- [ ] Run Section 1 (Shared Setup) before running any model section

### To run a single model

Each model has its own self-contained section. To run just one model:
1. Run Section 1 (Shared Setup) — cells 1.1 through 1.5
2. Run Section 2 (Data Loading) — cells 2.0 through 2.3
3. Run Section 3 (Shared Utilities) — cells 3.1 through 3.6
4. Jump to the model section you want and run it top to bottom

### Model overview

| Model | Type | Params | T4 time | A100 time |
|---|---|---|---|---|
| 🔵 DINOv2-Large | Self-supervised Vision Tfmr | 307M | ~90 min | ~25 min |
| 🟠 ConvNeXt V2-L | Convolutional Neural Net | 198M | ~120 min | ~35 min |
| 🟢 MedGemma-4B | Medical Vision-Language | 4B | ~180 min | ~50 min |
| 🟣 Qwen2.5-VL-7B | General Vision-Language | 7B | ~240 min | ~70 min |
| 🔴 LLaVA-Med-7B | Medical Vision-Language | 7B | ~240 min | ~70 min |
| ⚪ Qwen3-VL-8B | General Vision-Language | 8B | ~280 min | ~80 min |

Times shown are for 5 epochs on ~8,600 training images. T4 = free Colab GPU (8GB VRAM). A100 = Colab Pro / RunPod ($1.49/hr).

### The 8 diagnostic categories

- **MEL (0)** — Melanoma: The most dangerous skin cancer
- **NV (1)** — Melanocytic Nevus: Common mole (usually benign)
- **BCC (2)** — Basal Cell Carcinoma: Most common skin cancer, rarely fatal
- **AK (3)** — Actinic Keratosis: Pre-cancerous rough skin patch
- **BKL (4)** — Benign Keratosis: Harmless skin growth (seborrheic keratosis etc.)
- **DF (5)** — Dermatofibroma: Benign fibrous skin nodule
- **VASC (6)** — Vascular Lesion: Blood vessel growth (angioma etc.)
- **SCC (7)** — Squamous Cell Carcinoma: Second most common skin cancer

## ⚙️  SECTION 1 — SHARED SETUP

Run these cells ONCE at the start of every session. Every model section below depends on these.

### Cell 1.1 — Install required packages

This installs all Python libraries needed across all 6 models. ⏱️ Takes 2–4 minutes. Restart runtime when prompted after installation.

In [1]:
!pip install -q \
  transformers>=4.47.0 \
  peft>=0.12.0 \
  bitsandbytes>=0.43.3 \
  accelerate>=0.33.0 \
  timm==1.0.7 \
  scikit-learn \
  matplotlib \
  seaborn \
  pandas \
  Pillow \
  tqdm

# After installation completes: Runtime -> Restart session, then continue from Cell 1.2.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 41.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


### Cell 1.2 — Connect to Google Drive & get the code repo

Drive holds the actual data (`metadata.csv` + `derm_images/`, ~1.2GB — too
large for the git repo) and is where checkpoints/results get persisted
across Colab sessions. The repo itself (notebook + `scripts/`) is small and
gets cloned from GitHub for its code.

**Before running this cell**: upload `metadata.csv`, `derm_images/`,
`licenses/`, and `attribution.txt` to `MyDrive/BCN20000/` in your Google
Drive (edit `DATA_SOURCE_ROOT` in Cell 1.5 if you use a different folder).


In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive connected (data + checkpoint storage)")

    REPO_ROOT = Path("/content/derm-research")
    if not REPO_ROOT.exists():
        import subprocess
        subprocess.run(
            ["git", "clone", "https://github.com/ShonenMind/derm-research.git", str(REPO_ROOT)],
            check=True,
        )
    print(f"Code repo ready at {REPO_ROOT}")
else:
    # Already running from inside a local checkout of the repo
    REPO_ROOT = Path.cwd()
    print(f"Running locally — using repo at {REPO_ROOT}")


### Cell 1.3 — GPU check

Confirms a GPU is available and shows how much memory you have.

In [3]:
import torch

if torch.cuda.is_available():
    gpu_name  = torch.cuda.get_device_name(0)
    vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU detected  : {gpu_name}")
    print(f"✓ VRAM available: {vram_gb:.1f} GB")
    if vram_gb < 10:
        print("⚠️  Less than 10GB VRAM — use batch_size=8 for VLM models")
    elif vram_gb < 20:
        print("ℹ️  T4-class GPU — all models will fit; VLMs may need batch_size=4")
    else:
        print("✓ A100-class GPU — all models fit comfortably")
else:
    print("❌ No GPU found. Go to Runtime → Change runtime type → GPU")
    print("   Without a GPU this notebook will be extremely slow.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

✓ GPU detected  : Tesla T4
✓ VRAM available: 15.6 GB
ℹ️  T4-class GPU — all models will fit; VLMs may need batch_size=4


### Cell 1.4 — All imports

Loads every library used across all model sections.

In [6]:
import os
import json
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
from torchvision import transforms

# HuggingFace
from transformers import (
    AutoImageProcessor,
    AutoModelForImageTextToText,
    AutoProcessor,
    Dinov2Model,
    ConvNextV2ForImageClassification,
    LlavaForConditionalGeneration,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig,
)

# PEFT (for QLoRA on VLM models)
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

# Metrics
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print("✓ All imports loaded successfully")

ℹ️  Qwen2_5_VLForConditionalGeneration not found — update transformers if using Qwen models
✓ All imports loaded successfully


### Cell 1.5 — Global configuration

⚙️  EDIT THE PATHS HERE before running anything else.

In [ ]:
# ⚙️ SETTINGS ─────────────────────────────────────────────────────────────────

# Folder containing metadata.csv + derm_images/. On Colab this is a Google
# Drive folder you upload the data to (it does NOT live in the git repo —
# see Cell 1.2). Locally it's just the repo root, since the data already
# sits there.
DATA_SOURCE_ROOT = "/content/drive/MyDrive/BCN20000" if IN_COLAB else str(REPO_ROOT)

# Set to False once you've confirmed the Drive upload is complete — checking
# is_file() on ~19k images over a mounted Drive folder is slow; skipping it
# is safe after the first successful run.
VERIFY_IMAGE_FILES = True

# Where train.csv / val.csv / test.csv get written by Cell 2.0 below
# (connects metadata to images). Kept alongside the source data on Colab so
# it persists across sessions; a plain repo subfolder locally.
DATA_ROOT      = DATA_SOURCE_ROOT if IN_COLAB else str(REPO_ROOT / "data")

# Where model checkpoints and result files get saved. On Colab this lives on
# Drive so it survives runtime restarts; locally it's just a repo folder.
CHECKPOINT_DIR = "/content/drive/MyDrive/derm_checkpoints" if IN_COLAB else str(REPO_ROOT / "checkpoints")

# Reproducibility — do not change this between models
SEED = 42

# ─────────────────────────────────────────────────────────────────────────────

# Derived paths — no need to edit these
TRAIN_CSV = f"{DATA_ROOT}/train.csv"
VAL_CSV   = f"{DATA_ROOT}/val.csv"
TEST_CSV  = f"{DATA_ROOT}/test.csv"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Class definitions — must match the labels in your CSVs
CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
N_CLASSES   = len(CLASS_NAMES)

# Full names for display in plots and reports
CLASS_FULL_NAMES = [
    "Melanoma", "Melanocytic Nevus", "Basal Cell Carcinoma",
    "Actinic Keratosis", "Benign Keratosis", "Dermatofibroma",
    "Vascular Lesion", "Squamous Cell Carcinoma",
]

def set_seed(seed: int = 42) -> None:
    """Lock down all random number generators for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
print(f"✓ Seed set to {SEED}")
print(f"✓ Data source: {DATA_SOURCE_ROOT}")
print(f"✓ Data root  : {DATA_ROOT}")
print(f"✓ Checkpoints: {CHECKPOINT_DIR}")
print(f"✓ Classes    : {CLASS_NAMES}")


## 📂  SECTION 2 — DATA LOADING

Defines the dataset class and computes class weights. These are shared across all 6 models.

### THE BCN20000 DATASET

BCN20000 contains 19,424 dermoscopic photographs of skin lesions taken at the Hospital Clínic de Barcelona between 2010 and 2016. Unlike cleaner benchmark datasets, BCN20000 was designed to mirror real clinical practice — including lesions on nails and mucosa, large lesions that don't fit the dermoscope aperture, and hard-to-diagnose hypopigmented lesions.

The training set (~8,600 images after our split) has significant class imbalance: over half the images show benign moles (NV), while rare classes like Dermatofibroma (DF) and Vascular Lesion (VASC) have only ~80–150 images. We handle this with class-weighted loss so the model can't just predict "mole" for everything and get acceptable accuracy.

### Cell 2.0 — Connect metadata to images and build train/val/test splits

The images in `derm_images/` and the rows in `metadata.csv` (both under
`DATA_SOURCE_ROOT`, see Cell 1.5) are joined here by `isic_id`. Two problems
get handled before anything is split:

1. **Metadata ↔ image linkage** — `metadata.csv` has no path column, so an
   `image_path` column is built pointing at `derm_images/<isic_id>.jpg`.
2. **Unlabeled / unusable images are dropped** — 1,156 images have no
   diagnosis at all, and 314 are labeled "Scar", which isn't one of the 8
   target diagnostic classes (`CLASS_NAMES` above). Only the remaining
   ~17.5k labeled images are used to build the stratified 70/15/15
   train/val/test split. The full label-mapping logic lives in
   `scripts/prepare_data.py` (imported below, not duplicated here).


In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from prepare_data import build_connected_dataframe, split_and_save, CLASS_NAMES as _PREP_CLASS_NAMES

assert _PREP_CLASS_NAMES == CLASS_NAMES, "CLASS_NAMES in prepare_data.py must match Cell 1.5"

connected_df = build_connected_dataframe(Path(DATA_SOURCE_ROOT), verify_files=VERIFY_IMAGE_FILES)
split_and_save(connected_df, Path(DATA_ROOT), seed=SEED)


### Cell 2.1 — BCN20000 Dataset class

This class reads image paths and labels from the CSV files produced by the data preparation script and feeds them to the AI models. It supports two modes: one for standard vision models, one for VLMs.

In [ ]:
class BCN20000Dataset(Dataset):
    """
    Universal dataset class for BCN20000 dermoscopy images.

    mode = "vision"
        Returns (pixel_values_tensor, label_int)
        Used by: DINOv2-Large, ConvNeXt V2-Large

    mode = "vlm"
        Returns (pil_image, prompt_string, label_int)
        Used by: MedGemma-4B, Qwen2.5-VL-7B, LLaVA-Med-7B, Qwen3-VL-8B
    """

    # Prompt used by all 4 VLM models — kept here so it's easy to update
    VLM_SYSTEM_PROMPT = (
        "You are a dermatology AI assistant. Your task is to classify a "
        "dermoscopic skin lesion image into exactly one diagnostic category. "
        "The 8 possible categories are: "
        "MEL (Melanoma), NV (Melanocytic Nevus / mole), "
        "BCC (Basal Cell Carcinoma), AK (Actinic Keratosis), "
        "BKL (Benign Keratosis), DF (Dermatofibroma), "
        "VASC (Vascular Lesion), SCC (Squamous Cell Carcinoma). "
        "Respond with ONLY the 2-4 letter abbreviation. Nothing else."
    )
    VLM_USER_PROMPT = "Classify this dermoscopic image."

    def __init__(
        self,
        csv_path: str,
        processor,                    # HuggingFace processor or None for VLM raw mode
        mode: str = "vision",         # "vision" or "vlm"
        augment: bool = False,        # only applies to vision mode
    ):
        self.df        = pd.read_csv(csv_path)
        self.processor = processor
        self.mode      = mode
        self.augment   = augment

        # Dermoscopy-appropriate augmentation for training
        # Lesions appear at any orientation → flips and rotations are safe
        # ColorJitter accounts for variability between dermoscope devices
        if augment and mode == "vision":
            self.aug = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.RandomRotation(degrees=45),
                transforms.ColorJitter(
                    brightness=0.25, contrast=0.25,
                    saturation=0.15, hue=0.05,
                ),
            ])
        else:
            self.aug = None

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row   = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])

        if self.mode == "vision":
            if self.aug is not None:
                image = self.aug(image)
            inputs       = self.processor(images=image, return_tensors="pt")
            pixel_values = inputs["pixel_values"].squeeze(0)
            return pixel_values, torch.tensor(label, dtype=torch.long)

        elif self.mode == "vlm":
            # Return raw PIL image + prompt text + label
            # Each VLM model's training loop handles tokenisation differently
            return image, self.VLM_USER_PROMPT, torch.tensor(label, dtype=torch.long)

        else:
            raise ValueError(f"Unknown mode '{self.mode}'. Use 'vision' or 'vlm'.")

    def get_stratification_columns(self, idx: int) -> dict:
        """Returns metadata for stratified analysis (used in evaluation)."""
        row = self.df.iloc[idx]
        return {
            "diagnosis_1" : row.get("diagnosis_1", None),
            "melanocytic" : row.get("melanocytic", None),
            "anatom_site" : row.get("anatom_site_general", None),
            "age_approx"  : row.get("age_approx", None),
            "sex"         : row.get("sex", None),
        }

print("✓ BCN20000Dataset class defined")

### Cell 2.2 — Class imbalance visualisation

Shows how many images exist per diagnostic category in the training set. Classes with very few images need extra attention during training.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
class_counts = train_df["label"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
colors  = ["#e74c3c" if c < 300 else "#3498db" for c in class_counts.values]
bars    = ax.barh(CLASS_FULL_NAMES, class_counts.values, color=colors, edgecolor="white")

for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            str(count), va="center", fontsize=9)

ax.set_xlabel("Number of training images")
ax.set_title("BCN20000 Training Set — Class Distribution\n"
             "(red bars = < 300 images — these are the hardest classes to learn)",
             fontsize=11)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/class_distribution.png", dpi=150)
plt.show()

for i, (name, count) in enumerate(zip(CLASS_FULL_NAMES, class_counts.values)):
    flag = " ⚠️  LOW" if count < 300 else ""
    print(f"  [{i}] {CLASS_NAMES[i]:<6} {name:<30} {count:>5} images{flag}")

### Cell 2.3 — Compute class weights

Rare classes get higher weight so the model can't ignore them. These weights are passed into the loss function during training.

In [ ]:
def compute_class_weights(csv_path: str, n_classes: int) -> torch.Tensor:
    """
    Inverse-frequency weighting: a class with half the images of another
    gets twice the penalty when the model gets it wrong.
    """
    df      = pd.read_csv(csv_path)
    counts  = df["label"].value_counts().sort_index()
    weights = 1.0 / counts.values.astype(float)
    weights = weights / weights.sum() * n_classes   # normalise so sum = n_classes
    return torch.tensor(weights, dtype=torch.float32)

CLASS_WEIGHTS = compute_class_weights(TRAIN_CSV, N_CLASSES).to(device)

print("Class weights (higher = rarer class, gets more focus during training):")
for i, (name, w) in enumerate(zip(CLASS_NAMES, CLASS_WEIGHTS)):
    bar = "█" * int(w.item() * 5)
    print(f"  {name:<6} {bar:<25} {w.item():.3f}x")

## 🔧  SECTION 3 — SHARED TRAINING UTILITIES

Used by all 6 models. Do not modify unless you know what you are doing.

### Cell 3.1 — Mixup augmentation helper

Mixup blends two images together during training. This helps the model learn smoother decision boundaries and is especially useful when some classes have very few examples.

In [ ]:
def mixup_batch(
    pixels: torch.Tensor,
    labels: torch.Tensor,
    n_classes: int,
    alpha: float = 0.2,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Returns mixed images and soft (blended) labels."""
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(pixels.size(0), device=pixels.device)
    mixed = lam * pixels + (1 - lam) * pixels[idx]
    y_a   = F.one_hot(labels,      n_classes).float()
    y_b   = F.one_hot(labels[idx], n_classes).float()
    return mixed, lam * y_a + (1 - lam) * y_b


def soft_cross_entropy(
    logits: torch.Tensor,
    soft_labels: torch.Tensor,
    class_weights: torch.Tensor | None = None,
) -> torch.Tensor:
    """Cross-entropy loss that works with soft/mixed labels."""
    log_probs = F.log_softmax(logits, dim=1)
    if class_weights is not None:
        log_probs = log_probs * class_weights.unsqueeze(0)
    return -(soft_labels * log_probs).sum(dim=1).mean()

### Cell 3.2 — VLM output parser

After a VLM generates text output (e.g. "MEL" or "I think it's melanoma"), this function extracts the class label. If it can't, it logs the failure.

In [ ]:
def parse_vlm_output(text: str, class_names: list[str]) -> int:
    """
    Extracts a class label from free-form VLM text output.

    Strategy:
      1. Check if the first word exactly matches a class abbreviation
      2. Scan full output for any abbreviation substring
      3. Return -1 (parse failure) if nothing found
    """
    text = text.strip().upper()
    # Strategy 1: exact first token match
    first_token = text.split()[0] if text.split() else ""
    if first_token in class_names:
        return class_names.index(first_token)
    # Strategy 2: substring scan
    for i, name in enumerate(class_names):
        if name in text:
            return i
    return -1   # parse failure

### Cell 3.3 — Training loop (vision models)

Runs one complete pass through the training data for DINOv2 and ConvNeXt.

In [ ]:
def train_epoch_vision(
    model,
    loader: DataLoader,
    optimizer,
    scaler: GradScaler,
    class_weights: torch.Tensor,
    grad_accum: int,
    use_mixup: bool,
    mixed_precision: bool,
) -> tuple[float, float, float]:
    """One training epoch for standard vision models. Returns loss, balanced_acc, macro_f1."""
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    optimizer.zero_grad()

    pbar = tqdm(enumerate(loader), total=len(loader), desc="  Training", leave=False)
    for step, (pixels, labels) in pbar:
        pixels, labels = pixels.to(device), labels.to(device)

        if use_mixup:
            pixels, soft_labels = mixup_batch(pixels, labels, N_CLASSES)
            soft_labels = soft_labels.to(device)
        else:
            soft_labels = F.one_hot(labels, N_CLASSES).float()

        with autocast("cuda", enabled=mixed_precision):
            out = model(pixel_values=pixels)
            logits = out.logits if hasattr(out, "logits") else out
            loss = soft_cross_entropy(logits, soft_labels, class_weights) / grad_accum

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss  += loss.item() * grad_accum * labels.size(0)
        preds        = logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(soft_labels.argmax(dim=1).cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item() * grad_accum:.4f}")

    n           = len(all_labels)
    epoch_loss  = total_loss / n
    bal_acc     = balanced_accuracy_score(all_labels, all_preds)
    macro_f1    = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return epoch_loss, bal_acc, macro_f1

### Cell 3.4 — Evaluation loop (vision models)

Runs the model on validation or test data without updating weights.

In [ ]:
@torch.no_grad()
def eval_epoch_vision(
    model,
    loader: DataLoader,
    class_weights: torch.Tensor,
    mixed_precision: bool,
) -> tuple:
    """Returns: loss, balanced_acc, macro_f1, y_true, y_pred, y_prob"""
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []

    for pixels, labels in tqdm(loader, desc="  Evaluating", leave=False):
        pixels, labels = pixels.to(device), labels.to(device)
        soft_labels    = F.one_hot(labels, N_CLASSES).float()

        with autocast("cuda", enabled=mixed_precision):
            try:
                out    = model(pixel_values=pixels)
                logits = out.logits if hasattr(out, "logits") else out
            except TypeError:
                logits = model(pixels)
            loss = soft_cross_entropy(logits, soft_labels, class_weights)

        total_loss += loss.item() * labels.size(0)
        probs       = torch.softmax(logits, dim=1).cpu().numpy()
        preds       = probs.argmax(axis=1)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    n          = len(all_labels)
    epoch_loss = total_loss / n
    bal_acc    = balanced_accuracy_score(all_labels, all_preds)
    macro_f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return (epoch_loss, bal_acc, macro_f1,
            np.array(all_labels), np.array(all_preds), np.array(all_probs))

### Cell 3.5 — Main training loop (shared by all models)

Handles early stopping, checkpoint saving, and prints a clean summary after each epoch so you can monitor progress.

In [ ]:
def run_training_loop(
    model,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer,
    scheduler,
    scaler: GradScaler,
    checkpoint_path: str,
    model_label: str,
    num_epochs: int,
    patience: int,
    class_weights: torch.Tensor,
    grad_accum: int = 2,
    use_mixup: bool = False,
    mixed_precision: bool = True,
) -> dict:
    """
    Runs the full training loop. Saves the best checkpoint automatically.
    Returns a history dict with per-epoch metrics for plotting.
    """
    history = {k: [] for k in ["train_loss", "val_loss",
                                "train_bal_acc", "val_bal_acc",
                                "train_f1", "val_f1"]}
    best_val_bal_acc  = 0.0
    epochs_no_improve = 0

    print(f"\n{'═'*60}")
    print(f"  {model_label}")
    print(f"  Epochs: {num_epochs}  |  Early stop patience: {patience}")
    print(f"{'═'*60}\n")
    print(f"  {'Epoch':<8} {'Tr Loss':<10} {'Tr BalAcc':<12} {'Vl Loss':<10} {'Vl BalAcc':<12} {'Vl F1'}")
    print(f"  {'─'*58}")

    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_bal, tr_f1 = train_epoch_vision(
            model, train_loader, optimizer, scaler,
            class_weights, grad_accum, use_mixup, mixed_precision,
        )
        if scheduler is not None:
            scheduler.step()

        vl_loss, vl_bal, vl_f1, _, _, _ = eval_epoch_vision(
            model, val_loader, class_weights, mixed_precision,
        )

        for k, v in zip(history.keys(),
                        [tr_loss, vl_loss, tr_bal, vl_bal, tr_f1, vl_f1]):
            history[k].append(v)

        marker = " ★" if vl_bal > best_val_bal_acc else ""
        print(f"  {epoch:<8} {tr_loss:<10.4f} {tr_bal:<12.4f} "
              f"{vl_loss:<10.4f} {vl_bal:<12.4f} {vl_f1:.4f}{marker}")

        if vl_bal > best_val_bal_acc:
            best_val_bal_acc  = vl_bal
            epochs_no_improve = 0
            torch.save({
                "epoch"            : epoch,
                "model_state_dict" : model.state_dict(),
                "val_bal_acc"      : vl_bal,
                "val_f1"           : vl_f1,
                "class_names"      : CLASS_NAMES,
            }, checkpoint_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"\n  Early stopping at epoch {epoch} "
                      f"(no improvement for {patience} epochs)")
                break

    print(f"\n  ✓ Best val balanced accuracy: {best_val_bal_acc:.4f}")
    print(f"  ✓ Checkpoint saved: {checkpoint_path}")
    return history

### Cell 3.6 — LR schedule builder

Cosine annealing with a warmup phase — the standard schedule for fine-tuning large pretrained models. Learning rate ramps up gently then decays smoothly.

In [ ]:
def build_cosine_schedule(optimizer, num_epochs: int, steps_per_epoch: int,
                           warmup_epochs: int = 1, grad_accum: int = 2):
    """Cosine LR decay with linear warmup."""
    total_steps  = (num_epochs * steps_per_epoch) // grad_accum
    warmup_steps = (warmup_epochs * steps_per_epoch) // grad_accum

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

## 📊  SECTION 4 — SHARED EVALUATION UTILITIES

Load a saved checkpoint and produce all paper-ready outputs. Call these after training any model.

### Cell 4.1 — Training curve plotter

Saves a loss / balanced accuracy / F1 plot to your Drive after training.

In [ ]:
def plot_training_curves(history: dict, model_label: str, save_dir: str) -> None:
    """Saves a 3-panel training curve figure as PNG."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    panels = [
        ("Loss",              "train_loss",    "val_loss"),
        ("Balanced Accuracy", "train_bal_acc", "val_bal_acc"),
        ("Macro F1",          "train_f1",      "val_f1"),
    ]
    for ax, (title, tr_key, vl_key) in zip(axes, panels):
        ax.plot(epochs, history[tr_key], label="Train", linewidth=2)
        ax.plot(epochs, history[vl_key], label="Val",   linewidth=2, linestyle="--")
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)

    plt.suptitle(f"{model_label} — Training Curves", fontsize=13, y=1.02)
    plt.tight_layout()
    path = os.path.join(save_dir, f"{model_label.lower().replace(' ', '_')}_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✓ Curves saved: {path}")

### Cell 4.2 — Full test set evaluation

Loads the best checkpoint, runs on test data, and saves all results.

In [ ]:
def run_full_evaluation(
    model,
    test_loader: DataLoader,
    checkpoint_path: str,
    model_label: str,
    save_dir: str,
    class_weights: torch.Tensor,
    mixed_precision: bool = True,
) -> dict:
    """
    Loads best checkpoint → runs test set → prints and saves all metrics.
    Returns a results dict for the final comparison table.
    """
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"  Loaded checkpoint from epoch {ckpt['epoch']} "
          f"(val bal_acc={ckpt['val_bal_acc']:.4f})\n")

    _, bal_acc, macro_f1, y_true, y_pred, y_prob = eval_epoch_vision(
        model, test_loader, class_weights, mixed_precision,
    )

    # AUROC
    try:
        auroc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auroc = float("nan")

    print(f"  {'─'*50}")
    print(f"  TEST RESULTS — {model_label}")
    print(f"  {'─'*50}")
    print(f"  Balanced Accuracy : {bal_acc:.4f}  ← primary metric")
    print(f"  Macro F1          : {macro_f1:.4f}")
    print(f"  Macro AUROC       : {auroc:.4f}")
    print(f"\n  Per-class report:")
    print(classification_report(y_true, y_pred,
                                target_names=CLASS_NAMES, digits=4))

    # Save classification report to CSV
    report_dict = classification_report(y_true, y_pred,
                                        target_names=CLASS_NAMES,
                                        output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_path = os.path.join(save_dir, f"{model_label.lower().replace(' ', '_')}_report.csv")
    report_df.to_csv(report_path)
    print(f"  ✓ Report saved: {report_path}")

    results = {
        "model"        : model_label,
        "balanced_acc" : round(bal_acc,  4),
        "macro_f1"     : round(macro_f1, 4),
        "macro_auroc"  : round(auroc,    4),
        "best_epoch"   : ckpt["epoch"],
    }
    # Add per-class sensitivity from the report
    for cls in CLASS_NAMES:
        if cls in report_dict:
            results[f"sens_{cls}"] = round(report_dict[cls]["recall"], 4)

    results_path = os.path.join(save_dir,
                                f"{model_label.lower().replace(' ', '_')}_results.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)

    return results

### Cell 4.3 — Confusion matrix plotter

Shows where the model confuses one diagnosis for another — the most clinically informative diagnostic output.

In [ ]:
def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    model_label: str,
    save_dir: str,
    cmap: str = "Blues",
) -> None:
    """Saves a normalised confusion matrix heatmap as PNG."""
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(
        cm_norm, annot=True, fmt=".2f", cmap=cmap,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, ax=ax, vmin=0, vmax=1,
    )
    ax.set_xlabel("Predicted Diagnosis", fontsize=12)
    ax.set_ylabel("True Diagnosis",      fontsize=12)
    ax.set_title(f"{model_label}\nNormalised Confusion Matrix — BCN20000 Test Set",
                 fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    path = os.path.join(save_dir,
                        f"{model_label.lower().replace(' ', '_')}_confusion.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✓ Confusion matrix saved: {path}")

### Cell 4.4 — Stratified analysis

Breaks down performance by clinical subgroups — a key part of the paper. Answers: "Does the model perform differently on malignant vs benign lesions?"

In [ ]:
def run_stratified_analysis(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    test_csv: str,
    model_label: str,
    save_dir: str,
) -> None:
    """
    Stratified performance breakdown by:
      - diagnosis_1 (Benign / Malignant / Indeterminate)
      - melanocytic (True / False)
    Saves a summary CSV for the paper.
    """
    meta = pd.read_csv(test_csv)
    rows = []

    for col, label in [("diagnosis_1", "Malignancy category"),
                       ("melanocytic", "Melanocytic status")]:
        if col not in meta.columns:
            print(f"  ⚠️  Column '{col}' not in test CSV — skipping")
            continue

        print(f"\n  Stratified by {label} ({col}):")
        print(f"  {'Group':<30} {'N':<8} {'Bal.Acc':<12} {'Macro F1'}")
        print(f"  {'─'*55}")

        for group in meta[col].dropna().unique():
            mask    = (meta[col] == group).values
            if mask.sum() < 5:
                continue
            g_true  = y_true[mask]
            g_pred  = y_pred[mask]
            g_bal   = balanced_accuracy_score(g_true, g_pred)
            g_f1    = f1_score(g_true, g_pred, average="macro", zero_division=0)
            print(f"  {str(group):<30} {mask.sum():<8} {g_bal:<12.4f} {g_f1:.4f}")
            rows.append({
                "model": model_label, "stratum_col": col,
                "group": group, "n": mask.sum(),
                "balanced_acc": round(g_bal, 4),
                "macro_f1": round(g_f1, 4),
            })

    if rows:
        strat_df   = pd.DataFrame(rows)
        strat_path = os.path.join(
            save_dir,
            f"{model_label.lower().replace(' ', '_')}_stratified.csv"
        )
        strat_df.to_csv(strat_path, index=False)
        print(f"\n  ✓ Stratified analysis saved: {strat_path}")

## 🔵  SECTION 5 — MODEL 1: DINOv2-Large

**What is DINOv2?**

DINOv2 is a Vision Transformer trained by Meta AI using self-supervised learning — it learned to understand images by looking at millions of unlabelled photos, with no human annotation required. It produces exceptionally rich visual features that transfer well to medical images even though it was never trained on skin lesions.

**Why include it?**

It represents the "pure vision" paradigm: no language, no medical pretraining — just powerful general visual understanding. It typically sets the accuracy ceiling in dermoscopy classification benchmarks and serves as the baseline every other model must beat to justify its added complexity.

### Cell 5.1 — DINOv2 settings

⚙️  Adjust batch size and epochs here. All other settings are reasonable defaults — only change them if you know what you are doing.

In [ ]:
# ⚙️ SETTINGS ─────────────────────────────────────────────────────────────────
DINOV2_MODEL_ID    = "facebook/dinov2-large"  # 307M parameters
DINOV2_BATCH_SIZE  = 32    # reduce to 16 if you get an out-of-memory error
DINOV2_EPOCHS      = 10
DINOV2_LR_BACKBONE = 1e-5  # very low LR for the pretrained backbone
DINOV2_LR_HEAD     = 1e-3  # higher LR for the new classification head
DINOV2_WEIGHT_DECAY= 0.01
DINOV2_GRAD_ACCUM  = 2     # simulates batch_size * 2 = 64 effective batch
DINOV2_PATIENCE    = 5     # stop if val balanced acc doesn't improve for 5 epochs
DINOV2_CHECKPOINT  = f"{CHECKPOINT_DIR}/dinov2_large_best.pth"
DINOV2_RESULTS_DIR = f"{CHECKPOINT_DIR}/dinov2_large"
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DINOV2_RESULTS_DIR, exist_ok=True)

### Cell 5.2 — DINOv2 model definition

In [ ]:
class DINOv2Classifier(nn.Module):
    """
    DINOv2-Large backbone with a custom classification head.

    The backbone extracts a 1024-dimensional feature vector from the [CLS]
    token (a summary of the whole image). The head then maps this to 8
    diagnostic categories using a two-layer MLP.
    """

    def __init__(self, model_id: str, num_classes: int):
        super().__init__()
        self.backbone = Dinov2Model.from_pretrained(model_id)
        hidden_size   = self.backbone.config.hidden_size  # 1024 for -large

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 512),
            nn.GELU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        outputs   = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0, :]  # [B, 1024]
        return self.head(cls_token)                      # [B, N_CLASSES]

### Cell 5.3 — Load DINOv2 model and processor

Downloads weights from HuggingFace (~1.2 GB). Requires internet access. ⏱️ Download: ~3–5 minutes on first run. Cached afterwards.

In [ ]:
dinov2_processor = AutoImageProcessor.from_pretrained(DINOV2_MODEL_ID)
dinov2_model     = DINOv2Classifier(DINOV2_MODEL_ID, N_CLASSES).to(device)

total_p     = sum(p.numel() for p in dinov2_model.parameters())
trainable_p = sum(p.numel() for p in dinov2_model.parameters() if p.requires_grad)
print(f"  Model     : DINOv2-Large")
print(f"  Total params    : {total_p/1e6:.1f}M")
print(f"  Trainable params: {trainable_p/1e6:.1f}M")
print(f"  Input resolution: {dinov2_processor.size}")

### Cell 5.4 — DINOv2 DataLoaders

In [ ]:
dinov2_train_ds = BCN20000Dataset(TRAIN_CSV, dinov2_processor, mode="vision", augment=True)
dinov2_val_ds   = BCN20000Dataset(VAL_CSV,   dinov2_processor, mode="vision", augment=False)
dinov2_test_ds  = BCN20000Dataset(TEST_CSV,  dinov2_processor, mode="vision", augment=False)

dinov2_train_loader = DataLoader(dinov2_train_ds, batch_size=DINOV2_BATCH_SIZE,
                                  shuffle=True,  num_workers=2, pin_memory=True)
dinov2_val_loader   = DataLoader(dinov2_val_ds,   batch_size=DINOV2_BATCH_SIZE,
                                  shuffle=False, num_workers=2, pin_memory=True)
dinov2_test_loader  = DataLoader(dinov2_test_ds,  batch_size=DINOV2_BATCH_SIZE,
                                  shuffle=False, num_workers=2, pin_memory=True)

print(f"  Train: {len(dinov2_train_ds):,} | Val: {len(dinov2_val_ds):,} | Test: {len(dinov2_test_ds):,}")

### Cell 5.5 — DINOv2 optimizer and scheduler

Two learning rates: the backbone (pretrained weights) trains slowly, the new classification head trains quickly.

In [ ]:
dinov2_optimizer = optim.AdamW([
    {"params": dinov2_model.backbone.parameters(), "lr": DINOV2_LR_BACKBONE},
    {"params": dinov2_model.head.parameters(),     "lr": DINOV2_LR_HEAD},
], weight_decay=DINOV2_WEIGHT_DECAY)

dinov2_scheduler = build_cosine_schedule(
    dinov2_optimizer, DINOV2_EPOCHS,
    steps_per_epoch=len(dinov2_train_loader),
    grad_accum=DINOV2_GRAD_ACCUM,
)
dinov2_scaler = GradScaler("cuda")

print(f"  Backbone LR : {DINOV2_LR_BACKBONE}")
print(f"  Head LR     : {DINOV2_LR_HEAD}")
print(f"  Effective batch size: {DINOV2_BATCH_SIZE * DINOV2_GRAD_ACCUM}")

### Cell 5.6 — Train DINOv2

⏱️ Approximately 90 minutes on T4 / 25 minutes on A100 (10 epochs)

In [ ]:
dinov2_history = run_training_loop(
    model           = dinov2_model,
    train_loader    = dinov2_train_loader,
    val_loader      = dinov2_val_loader,
    optimizer       = dinov2_optimizer,
    scheduler       = dinov2_scheduler,
    scaler          = dinov2_scaler,
    checkpoint_path = DINOV2_CHECKPOINT,
    model_label     = "DINOv2-Large",
    num_epochs      = DINOV2_EPOCHS,
    patience        = DINOV2_PATIENCE,
    class_weights   = CLASS_WEIGHTS,
    grad_accum      = DINOV2_GRAD_ACCUM,
    use_mixup       = False,
)
plot_training_curves(dinov2_history, "DINOv2-Large", DINOV2_RESULTS_DIR)

### Cell 5.7 — Evaluate DINOv2 on test set

In [ ]:
dinov2_results = run_full_evaluation(
    model           = dinov2_model,
    test_loader     = dinov2_test_loader,
    checkpoint_path = DINOV2_CHECKPOINT,
    model_label     = "DINOv2-Large",
    save_dir        = DINOV2_RESULTS_DIR,
    class_weights   = CLASS_WEIGHTS,
)

# Re-run eval to get y_true / y_pred for confusion matrix
_, _, _, d2_y_true, d2_y_pred, _ = eval_epoch_vision(
    dinov2_model, dinov2_test_loader, CLASS_WEIGHTS, mixed_precision=True,
)
plot_confusion_matrix(d2_y_true, d2_y_pred, "DINOv2-Large",
                      DINOV2_RESULTS_DIR, cmap="Blues")
run_stratified_analysis(d2_y_true, d2_y_pred, TEST_CSV,
                        "DINOv2-Large", DINOV2_RESULTS_DIR)

## 🟠  SECTION 6 — MODEL 2: ConvNeXt V2-Large

**What is ConvNeXt V2?**

ConvNeXt V2 is a modern Convolutional Neural Network (CNN) — the same family of architecture that powered dermatology AI since the landmark 2017 Nature paper. Unlike DINOv2, it uses convolution operations rather than transformers, which makes it excellent at picking up local texture patterns (crucial for dermoscopy: pigment network, regression, globules).

**Why include it?**

It answers the question: "Does the newer transformer architecture (DINOv2) actually beat the best CNN for skin lesion classification?" It's also the most direct comparison to prior ISIC challenge winners.

### Cell 6.1 — ConvNeXt V2 settings

In [ ]:
# ⚙️ SETTINGS ─────────────────────────────────────────────────────────────────
CONVNEXT_MODEL_ID    = "facebook/convnextv2-large-22k-384"  # 198M params, 384px input
CONVNEXT_BATCH_SIZE  = 16   # 384px images are larger — use 16 on T4, 32 on A100
CONVNEXT_EPOCHS      = 10
CONVNEXT_LR_BACKBONE = 5e-6
CONVNEXT_LR_HEAD     = 5e-4
CONVNEXT_WEIGHT_DECAY= 0.05  # ConvNeXt V2 paper recommends 0.05
CONVNEXT_GRAD_ACCUM  = 4     # effective batch = 16 * 4 = 64
CONVNEXT_PATIENCE    = 5
CONVNEXT_USE_MIXUP   = True  # Mixup helps CNNs with class imbalance
CONVNEXT_MIXUP_ALPHA = 0.2
CONVNEXT_CHECKPOINT  = f"{CHECKPOINT_DIR}/convnextv2_large_best.pth"
CONVNEXT_RESULTS_DIR = f"{CHECKPOINT_DIR}/convnextv2_large"
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(CONVNEXT_RESULTS_DIR, exist_ok=True)

### Cell 6.2 — Load ConvNeXt V2 model and processor

⏱️ Download: ~800 MB, approximately 2–3 minutes on first run.

In [ ]:
convnext_processor = AutoImageProcessor.from_pretrained(CONVNEXT_MODEL_ID)
convnext_model     = ConvNextV2ForImageClassification.from_pretrained(
    CONVNEXT_MODEL_ID,
    num_labels=N_CLASSES,
    ignore_mismatched_sizes=True,  # replaces the ImageNet-22K head with ours
).to(device)

# Replace the default linear head with a two-layer MLP for better generalisation
in_features = convnext_model.config.hidden_sizes[-1]  # 1536 for -large
convnext_model.classifier = nn.Sequential(
    nn.Flatten(),
    nn.LayerNorm(in_features),
    nn.Linear(in_features, 512),
    nn.GELU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, N_CLASSES),
).to(device)

total_p     = sum(p.numel() for p in convnext_model.parameters())
trainable_p = sum(p.numel() for p in convnext_model.parameters() if p.requires_grad)
print(f"  Model           : ConvNeXt V2-Large")
print(f"  Total params    : {total_p/1e6:.1f}M")
print(f"  Trainable params: {trainable_p/1e6:.1f}M")
print(f"  Input resolution: {convnext_processor.size}")

### Cell 6.3 — ConvNeXt V2 DataLoaders

In [ ]:
convnext_train_ds = BCN20000Dataset(TRAIN_CSV, convnext_processor, mode="vision", augment=True)
convnext_val_ds   = BCN20000Dataset(VAL_CSV,   convnext_processor, mode="vision", augment=False)
convnext_test_ds  = BCN20000Dataset(TEST_CSV,  convnext_processor, mode="vision", augment=False)

convnext_train_loader = DataLoader(convnext_train_ds, batch_size=CONVNEXT_BATCH_SIZE,
                                    shuffle=True,  num_workers=2, pin_memory=True)
convnext_val_loader   = DataLoader(convnext_val_ds,   batch_size=CONVNEXT_BATCH_SIZE,
                                    shuffle=False, num_workers=2, pin_memory=True)
convnext_test_loader  = DataLoader(convnext_test_ds,  batch_size=CONVNEXT_BATCH_SIZE,
                                    shuffle=False, num_workers=2, pin_memory=True)

print(f"  Train: {len(convnext_train_ds):,} | Val: {len(convnext_val_ds):,} | Test: {len(convnext_test_ds):,}")

### Cell 6.4 — ConvNeXt V2 optimizer and scheduler

In [ ]:
backbone_params  = [p for n, p in convnext_model.named_parameters() if "classifier" not in n]
head_params      = list(convnext_model.classifier.parameters())

convnext_optimizer = optim.AdamW([
    {"params": backbone_params, "lr": CONVNEXT_LR_BACKBONE},
    {"params": head_params,     "lr": CONVNEXT_LR_HEAD},
], weight_decay=CONVNEXT_WEIGHT_DECAY)

convnext_scheduler = build_cosine_schedule(
    convnext_optimizer, CONVNEXT_EPOCHS,
    steps_per_epoch=len(convnext_train_loader),
    grad_accum=CONVNEXT_GRAD_ACCUM,
)
convnext_scaler = GradScaler("cuda")

### Cell 6.5 — Train ConvNeXt V2

⏱️ Approximately 120 minutes on T4 / 35 minutes on A100 (10 epochs)

In [ ]:
convnext_history = run_training_loop(
    model           = convnext_model,
    train_loader    = convnext_train_loader,
    val_loader      = convnext_val_loader,
    optimizer       = convnext_optimizer,
    scheduler       = convnext_scheduler,
    scaler          = convnext_scaler,
    checkpoint_path = CONVNEXT_CHECKPOINT,
    model_label     = "ConvNeXt V2-Large",
    num_epochs      = CONVNEXT_EPOCHS,
    patience        = CONVNEXT_PATIENCE,
    class_weights   = CLASS_WEIGHTS,
    grad_accum      = CONVNEXT_GRAD_ACCUM,
    use_mixup       = CONVNEXT_USE_MIXUP,
)
plot_training_curves(convnext_history, "ConvNeXt V2-Large", CONVNEXT_RESULTS_DIR)

### Cell 6.6 — Evaluate ConvNeXt V2 on test set

In [ ]:
convnext_results = run_full_evaluation(
    model           = convnext_model,
    test_loader     = convnext_test_loader,
    checkpoint_path = CONVNEXT_CHECKPOINT,
    model_label     = "ConvNeXt V2-Large",
    save_dir        = CONVNEXT_RESULTS_DIR,
    class_weights   = CLASS_WEIGHTS,
)
_, _, _, cx_y_true, cx_y_pred, _ = eval_epoch_vision(
    convnext_model, convnext_test_loader, CLASS_WEIGHTS, mixed_precision=True,
)
plot_confusion_matrix(cx_y_true, cx_y_pred, "ConvNeXt V2-Large",
                      CONVNEXT_RESULTS_DIR, cmap="Oranges")
run_stratified_analysis(cx_y_true, cx_y_pred, TEST_CSV,
                        "ConvNeXt V2-Large", CONVNEXT_RESULTS_DIR)

## 🟢  SECTION 7 — MODEL 3: MedGemma-4B

**What is MedGemma?**

MedGemma is Google's medical Vision-Language Model — an AI that can understand both images AND text together. Unlike DINOv2 and ConvNeXt which only look at images, MedGemma was explicitly trained on over 51,000 dermatology images covering 210 skin conditions, as well as radiology, pathology, and ophthalmology data.

**Why include it?**

It tests whether "medical pretraining" gives a real advantage over general-purpose vision models. It also represents the emerging paradigm of AI systems that can eventually explain their reasoning in plain English — the direction clinical AI is heading.

Technical note: MedGemma uses QLoRA fine-tuning (4-bit quantisation + LoRA adapters) which allows a 4-billion parameter model to run on consumer hardware by trading a small amount of precision for a large reduction in memory usage.

### Cell 7.1 — MedGemma settings

In [ ]:
# ⚙️ SETTINGS ─────────────────────────────────────────────────────────────────
MEDGEMMA_MODEL_ID   = "google/medgemma-4b-it"   # 4B parameters
MEDGEMMA_BATCH_SIZE = 4     # VLMs are much larger — must use small batches
MEDGEMMA_EPOCHS     = 5     # VLMs converge faster; fewer epochs needed
MEDGEMMA_LR         = 2e-4  # Single LR for QLoRA adapters
MEDGEMMA_GRAD_ACCUM = 8     # effective batch = 4 * 8 = 32
MEDGEMMA_PATIENCE   = 3
MEDGEMMA_LORA_RANK  = 16    # LoRA rank — higher = more capacity, more memory
MEDGEMMA_LORA_ALPHA = 32    # LoRA scaling factor (usually 2 * rank)
MEDGEMMA_CHECKPOINT = f"{CHECKPOINT_DIR}/medgemma_4b_best.pth"
MEDGEMMA_RESULTS_DIR= f"{CHECKPOINT_DIR}/medgemma_4b"
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(MEDGEMMA_RESULTS_DIR, exist_ok=True)

### Cell 7.2 — QLoRA configuration (shared pattern for all VLMs)

QLoRA = 4-bit quantisation + Low-Rank Adaptation. Lets a 4B model fit in ~10GB VRAM by compressing weights to 4-bit and only training small adapter matrices (LoRA) rather than updating all 4 billion parameters.

In [ ]:
medgemma_bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",         # NormalFloat4 — best for LLM weights
    bnb_4bit_compute_dtype    = torch.bfloat16, # compute in bfloat16 for stability
    bnb_4bit_use_double_quant = True,           # double quantisation saves extra memory
)

medgemma_lora_config = LoraConfig(
    r              = MEDGEMMA_LORA_RANK,
    lora_alpha     = MEDGEMMA_LORA_ALPHA,
    target_modules = ["q_proj", "v_proj", "k_proj", "o_proj"],  # attention layers
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
)

### Cell 7.3 — Load MedGemma model and processor

⏱️ Download: ~8 GB, approximately 10–15 minutes on first run. Note: requires accepting the MedGemma license on HuggingFace first. Visit: https://huggingface.co/google/medgemma-4b-it and click "Agree"

In [ ]:
medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)

medgemma_base = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_MODEL_ID,
    quantization_config = medgemma_bnb_config,
    device_map          = "auto",
    torch_dtype         = torch.bfloat16,
)
medgemma_base = prepare_model_for_kbit_training(medgemma_base)
medgemma_model = get_peft_model(medgemma_base, medgemma_lora_config)
medgemma_model.print_trainable_parameters()

### Cell 7.4 — MedGemma DataLoader and VLM collation

VLMs need a custom collate function because each sample contains an image (PIL), a text prompt, and a label — not a pre-processed tensor.

In [ ]:
medgemma_train_ds = BCN20000Dataset(TRAIN_CSV, processor=None, mode="vlm", augment=True)
medgemma_val_ds   = BCN20000Dataset(VAL_CSV,   processor=None, mode="vlm", augment=False)
medgemma_test_ds  = BCN20000Dataset(TEST_CSV,  processor=None, mode="vlm", augment=False)

def vlm_collate_fn(batch):
    """Collects PIL images, prompt strings, and labels into batch lists."""
    images  = [item[0] for item in batch]
    prompts = [item[1] for item in batch]
    labels  = torch.stack([item[2] for item in batch])
    return images, prompts, labels

medgemma_train_loader = DataLoader(medgemma_train_ds, batch_size=MEDGEMMA_BATCH_SIZE,
                                    shuffle=True,  collate_fn=vlm_collate_fn)
medgemma_val_loader   = DataLoader(medgemma_val_ds,   batch_size=MEDGEMMA_BATCH_SIZE,
                                    shuffle=False, collate_fn=vlm_collate_fn)
medgemma_test_loader  = DataLoader(medgemma_test_ds,  batch_size=MEDGEMMA_BATCH_SIZE,
                                    shuffle=False, collate_fn=vlm_collate_fn)

print(f"  Train: {len(medgemma_train_ds):,} | Val: {len(medgemma_val_ds):,} | Test: {len(medgemma_test_ds):,}")

### Cell 7.5 — VLM training and evaluation loops

VLMs work differently from vision models: they generate text ("MEL", "NV" etc.) rather than predicting a class index directly. We parse that text output back into a class label for metric computation.

In [ ]:
def train_epoch_vlm(model, processor, loader, optimizer, scaler,
                    system_prompt, grad_accum, mixed_precision):
    """One training epoch for VLM models using prompt-based classification."""
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    optimizer.zero_grad()

    pbar = tqdm(enumerate(loader), total=len(loader), desc="  Training", leave=False)
    for step, (images, prompts, labels) in pbar:
        # Format as conversation and tokenise
        conversations = [
            [
                {"role": "system",  "content": system_prompt},
                {"role": "user",    "content": [
                    {"type": "image", "image": img},
                    {"type": "text",  "text": prompt},
                ]},
                {"role": "assistant", "content": CLASS_NAMES[lbl.item()]},
            ]
            for img, prompt, lbl in zip(images, prompts, labels)
        ]

        inputs = processor.apply_chat_template(
            conversations, add_generation_prompt=False, return_tensors="pt",
        ).to(device)

        with autocast("cuda", enabled=mixed_precision, dtype=torch.bfloat16):
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss    = outputs.loss / grad_accum

        scaler.scale(loss).backward()
        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum * len(labels)
        # For train metrics, use label directly (we know ground truth)
        all_labels.extend(labels.numpy())
        all_preds.extend(labels.numpy())   # placeholder — train accuracy less meaningful for VLMs
        pbar.set_postfix(loss=f"{loss.item() * grad_accum:.4f}")

    return total_loss / max(len(all_labels), 1), 0.0, 0.0


@torch.no_grad()
def eval_epoch_vlm(model, processor, loader, system_prompt, mixed_precision):
    """Evaluation epoch for VLMs: generate predictions and parse class labels."""
    model.eval()
    all_preds, all_labels = [], []
    parse_failures = 0

    for images, prompts, labels in tqdm(loader, desc="  Evaluating", leave=False):
        conversations = [
            [
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": [
                    {"type": "image", "image": img},
                    {"type": "text",  "text": prompt},
                ]},
            ]
            for img, prompt in zip(images, prompts)
        ]

        inputs = processor.apply_chat_template(
            conversations, add_generation_prompt=True, return_tensors="pt",
        ).to(device)

        with autocast("cuda", enabled=mixed_precision, dtype=torch.bfloat16):
            generated = model.generate(
                **inputs,
                max_new_tokens = 8,    # class labels are short (MEL, NV, etc.)
                do_sample      = False,
                temperature    = None,
                top_p          = None,
            )

        for i, (gen, lbl) in enumerate(zip(generated, labels)):
            input_len = inputs["input_ids"].shape[1]
            new_tokens = gen[input_len:]
            text       = processor.decode(new_tokens, skip_special_tokens=True)
            pred       = parse_vlm_output(text, CLASS_NAMES)
            if pred == -1:
                parse_failures += 1
                pred = 1   # fallback to NV (majority class) to avoid NaN metrics
            all_preds.append(pred)
            all_labels.append(lbl.item())

    if parse_failures > 0:
        print(f"  ⚠️  Parse failures: {parse_failures}/{len(all_labels)} "
              f"({100*parse_failures/len(all_labels):.1f}%) — check model outputs")

    bal_acc  = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return 0.0, bal_acc, macro_f1, np.array(all_labels), np.array(all_preds), None

### Cell 7.6 — Train MedGemma

⏱️ Approximately 180 minutes on T4 / 50 minutes on A100 (5 epochs)

In [ ]:
medgemma_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, medgemma_model.parameters()),
    lr=MEDGEMMA_LR, weight_decay=0.01,
)
medgemma_scaler = GradScaler("cuda")

best_val_bal_acc   = 0.0
medgemma_history   = {"val_bal_acc": [], "val_f1": []}

print(f"\n{'═'*60}")
print(f"  MedGemma-4B — BCN20000 — {N_CLASSES} classes")
print(f"{'═'*60}\n")

for epoch in range(1, MEDGEMMA_EPOCHS + 1):
    print(f"  Epoch {epoch}/{MEDGEMMA_EPOCHS}")
    train_epoch_vlm(
        medgemma_model, medgemma_processor, medgemma_train_loader,
        medgemma_optimizer, medgemma_scaler,
        BCN20000Dataset.VLM_SYSTEM_PROMPT,
        MEDGEMMA_GRAD_ACCUM, mixed_precision=True,
    )
    _, vl_bal, vl_f1, _, _, _ = eval_epoch_vlm(
        medgemma_model, medgemma_processor, medgemma_val_loader,
        BCN20000Dataset.VLM_SYSTEM_PROMPT, mixed_precision=True,
    )
    medgemma_history["val_bal_acc"].append(vl_bal)
    medgemma_history["val_f1"].append(vl_f1)
    print(f"  Val Balanced Acc: {vl_bal:.4f}  |  Val Macro F1: {vl_f1:.4f}")

    if vl_bal > best_val_bal_acc:
        best_val_bal_acc = vl_bal
        medgemma_model.save_pretrained(MEDGEMMA_RESULTS_DIR + "/best_lora")
        print(f"  ★ New best: {best_val_bal_acc:.4f} — LoRA adapters saved")

### Cell 7.7 — Evaluate MedGemma on test set

In [ ]:
_, mg_bal, mg_f1, mg_y_true, mg_y_pred, _ = eval_epoch_vlm(
    medgemma_model, medgemma_processor, medgemma_test_loader,
    BCN20000Dataset.VLM_SYSTEM_PROMPT, mixed_precision=True,
)
print(f"\n  TEST — MedGemma-4B")
print(f"  Balanced Accuracy : {mg_bal:.4f}")
print(f"  Macro F1          : {mg_f1:.4f}")
print(classification_report(mg_y_true, mg_y_pred, target_names=CLASS_NAMES, digits=4))
plot_confusion_matrix(mg_y_true, mg_y_pred, "MedGemma-4B",
                      MEDGEMMA_RESULTS_DIR, cmap="Greens")
run_stratified_analysis(mg_y_true, mg_y_pred, TEST_CSV,
                        "MedGemma-4B", MEDGEMMA_RESULTS_DIR)

## 🟣  SECTION 8 — MODEL 4: Qwen2.5-VL-7B

**What is Qwen2.5-VL?**

Qwen2.5-VL is a 7-billion parameter Vision-Language Model developed by Alibaba. It was trained on a massive general-purpose dataset of images and text — NOT specifically on medical data. It is currently one of the top-performing open-source VLMs on standard vision benchmarks.

**Why include it?**

It answers the key question: "Does a powerful general AI — with no medical pretraining — match or beat AI systems specifically designed for medicine, after both are fine-tuned on the same dataset?" This comparison is the core scientific contribution of the paper.

### Cell 8.1 — Qwen2.5-VL settings

In [ ]:
# ⚙️ SETTINGS ─────────────────────────────────────────────────────────────────
QWEN25_MODEL_ID    = "Qwen/Qwen2.5-VL-7B-Instruct"  # 7B parameters
QWEN25_BATCH_SIZE  = 4     # reduce to 2 if OOM on T4
QWEN25_EPOCHS      = 5
QWEN25_LR          = 2e-4
QWEN25_GRAD_ACCUM  = 8
QWEN25_PATIENCE    = 3
QWEN25_LORA_RANK   = 16
QWEN25_LORA_ALPHA  = 32
QWEN25_CHECKPOINT  = f"{CHECKPOINT_DIR}/qwen25_vl_7b_best"
QWEN25_RESULTS_DIR = f"{CHECKPOINT_DIR}/qwen25_vl_7b"
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(QWEN25_RESULTS_DIR, exist_ok=True)

### Cell 8.2 — Load Qwen2.5-VL model

⏱️ Download: ~15 GB, approximately 20 minutes on first run.

In [ ]:
qwen25_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
qwen25_lora_config = LoraConfig(
    r=QWEN25_LORA_RANK, lora_alpha=QWEN25_LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "up_proj", "down_proj", "gate_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)

qwen25_processor = AutoProcessor.from_pretrained(QWEN25_MODEL_ID)
qwen25_base      = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN25_MODEL_ID,
    quantization_config=qwen25_bnb_config,
    device_map="auto", torch_dtype=torch.bfloat16,
)
qwen25_base  = prepare_model_for_kbit_training(qwen25_base)
qwen25_model = get_peft_model(qwen25_base, qwen25_lora_config)
qwen25_model.print_trainable_parameters()

### Cell 8.3 — Qwen2.5-VL DataLoaders

In [ ]:
qwen25_train_ds = BCN20000Dataset(TRAIN_CSV, processor=None, mode="vlm", augment=True)
qwen25_val_ds   = BCN20000Dataset(VAL_CSV,   processor=None, mode="vlm", augment=False)
qwen25_test_ds  = BCN20000Dataset(TEST_CSV,  processor=None, mode="vlm", augment=False)

qwen25_train_loader = DataLoader(qwen25_train_ds, batch_size=QWEN25_BATCH_SIZE,
                                  shuffle=True,  collate_fn=vlm_collate_fn)
qwen25_val_loader   = DataLoader(qwen25_val_ds,   batch_size=QWEN25_BATCH_SIZE,
                                  shuffle=False, collate_fn=vlm_collate_fn)
qwen25_test_loader  = DataLoader(qwen25_test_ds,  batch_size=QWEN25_BATCH_SIZE,
                                  shuffle=False, collate_fn=vlm_collate_fn)

### Cell 8.4 — Train Qwen2.5-VL

⏱️ Approximately 240 minutes on T4 / 70 minutes on A100 (5 epochs)

In [ ]:
qwen25_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, qwen25_model.parameters()),
    lr=QWEN25_LR, weight_decay=0.01,
)
qwen25_scaler  = GradScaler("cuda")
best_q25_acc   = 0.0

print(f"\n{'═'*60}\n  Qwen2.5-VL-7B — BCN20000\n{'═'*60}\n")

for epoch in range(1, QWEN25_EPOCHS + 1):
    print(f"  Epoch {epoch}/{QWEN25_EPOCHS}")
    train_epoch_vlm(qwen25_model, qwen25_processor, qwen25_train_loader,
                    qwen25_optimizer, qwen25_scaler,
                    BCN20000Dataset.VLM_SYSTEM_PROMPT,
                    QWEN25_GRAD_ACCUM, mixed_precision=True)
    _, vl_bal, vl_f1, _, _, _ = eval_epoch_vlm(
        qwen25_model, qwen25_processor, qwen25_val_loader,
        BCN20000Dataset.VLM_SYSTEM_PROMPT, mixed_precision=True)
    print(f"  Val Balanced Acc: {vl_bal:.4f}  |  Val Macro F1: {vl_f1:.4f}")
    if vl_bal > best_q25_acc:
        best_q25_acc = vl_bal
        qwen25_model.save_pretrained(QWEN25_CHECKPOINT)
        print(f"  ★ New best: {best_q25_acc:.4f} — saved")

### Cell 8.5 — Evaluate Qwen2.5-VL on test set

In [ ]:
_, q25_bal, q25_f1, q25_y_true, q25_y_pred, _ = eval_epoch_vlm(
    qwen25_model, qwen25_processor, qwen25_test_loader,
    BCN20000Dataset.VLM_SYSTEM_PROMPT, mixed_precision=True)
print(f"\n  TEST — Qwen2.5-VL-7B")
print(f"  Balanced Accuracy : {q25_bal:.4f}  |  Macro F1: {q25_f1:.4f}")
print(classification_report(q25_y_true, q25_y_pred, target_names=CLASS_NAMES, digits=4))
plot_confusion_matrix(q25_y_true, q25_y_pred, "Qwen2.5-VL-7B",
                      QWEN25_RESULTS_DIR, cmap="Purples")
run_stratified_analysis(q25_y_true, q25_y_pred, TEST_CSV,
                        "Qwen2.5-VL-7B", QWEN25_RESULTS_DIR)

## 🔴  SECTION 9 — MODEL 5: LLaVA-Med-7B

**What is LLaVA-Med?**

LLaVA-Med is a medical Vision-Language Model from Microsoft, built by fine-tuning the general LLaVA model on 600,000 biomedical image-text pairs from PubMed Central. It is one of the most widely cited open medical VLMs and serves as an important comparison point.

**Why include it?**

It represents "older generation" medical VLM (2023 architecture vs MedGemma's 2024 architecture), which lets the paper answer: "Has medical VLM performance improved meaningfully in two years?" Its known weakness — strong class bias toward majority classes — also provides an important cautionary finding for the paper.

### Cell 9.1 — LLaVA-Med settings

In [ ]:
# ⚙️ SETTINGS ─────────────────────────────────────────────────────────────────
LLAVA_MODEL_ID    = "microsoft/llava-med-v1.5-mistral-7b"  # 7B parameters
LLAVA_BATCH_SIZE  = 4
LLAVA_EPOCHS      = 5
LLAVA_LR          = 2e-4
LLAVA_GRAD_ACCUM  = 8
LLAVA_PATIENCE    = 3
LLAVA_LORA_RANK   = 16
LLAVA_LORA_ALPHA  = 32
LLAVA_CHECKPOINT  = f"{CHECKPOINT_DIR}/llava_med_7b_best"
LLAVA_RESULTS_DIR = f"{CHECKPOINT_DIR}/llava_med_7b"
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(LLAVA_RESULTS_DIR, exist_ok=True)

### Cell 9.2 — Load LLaVA-Med model

⏱️ Download: ~14 GB, approximately 15–20 minutes on first run.

In [ ]:
llava_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
llava_lora_config = LoraConfig(
    r=LLAVA_LORA_RANK, lora_alpha=LLAVA_LORA_ALPHA,
    target_modules=["q_proj", "v_proj"],  # LLaVA-Med uses fewer target modules
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)

llava_processor = AutoProcessor.from_pretrained(LLAVA_MODEL_ID)
llava_base      = LlavaForConditionalGeneration.from_pretrained(
    LLAVA_MODEL_ID,
    quantization_config=llava_bnb_config,
    device_map="auto", torch_dtype=torch.bfloat16,
)
llava_base  = prepare_model_for_kbit_training(llava_base)
llava_model = get_peft_model(llava_base, llava_lora_config)
llava_model.print_trainable_parameters()

### Cell 9.3 — LLaVA-Med DataLoaders and training

⏱️ Approximately 240 minutes on T4 / 70 minutes on A100 (5 epochs)

In [ ]:
llava_train_ds = BCN20000Dataset(TRAIN_CSV, processor=None, mode="vlm", augment=True)
llava_val_ds   = BCN20000Dataset(VAL_CSV,   processor=None, mode="vlm", augment=False)
llava_test_ds  = BCN20000Dataset(TEST_CSV,  processor=None, mode="vlm", augment=False)

llava_train_loader = DataLoader(llava_train_ds, batch_size=LLAVA_BATCH_SIZE,
                                 shuffle=True,  collate_fn=vlm_collate_fn)
llava_val_loader   = DataLoader(llava_val_ds,   batch_size=LLAVA_BATCH_SIZE,
                                 shuffle=False, collate_fn=vlm_collate_fn)
llava_test_loader  = DataLoader(llava_test_ds,  batch_size=LLAVA_BATCH_SIZE,
                                 shuffle=False, collate_fn=vlm_collate_fn)

llava_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, llava_model.parameters()),
    lr=LLAVA_LR, weight_decay=0.01,
)
llava_scaler = GradScaler("cuda")
best_lv_acc  = 0.0

print(f"\n{'═'*60}\n  LLaVA-Med-7B — BCN20000\n{'═'*60}\n")

for epoch in range(1, LLAVA_EPOCHS + 1):
    print(f"  Epoch {epoch}/{LLAVA_EPOCHS}")
    train_epoch_vlm(llava_model, llava_processor, llava_train_loader,
                    llava_optimizer, llava_scaler,
                    BCN20000Dataset.VLM_SYSTEM_PROMPT,
                    LLAVA_GRAD_ACCUM, mixed_precision=True)
    _, vl_bal, vl_f1, _, _, _ = eval_epoch_vlm(
        llava_model, llava_processor, llava_val_loader,
        BCN20000Dataset.VLM_SYSTEM_PROMPT, mixed_precision=True)
    print(f"  Val Balanced Acc: {vl_bal:.4f}  |  Val Macro F1: {vl_f1:.4f}")
    if vl_bal > best_lv_acc:
        best_lv_acc = vl_bal
        llava_model.save_pretrained(LLAVA_CHECKPOINT)
        print(f"  ★ New best: {best_lv_acc:.4f} — saved")

### Cell 9.4 — Evaluate LLaVA-Med on test set

In [ ]:
_, lv_bal, lv_f1, lv_y_true, lv_y_pred, _ = eval_epoch_vlm(
    llava_model, llava_processor, llava_test_loader,
    BCN20000Dataset.VLM_SYSTEM_PROMPT, mixed_precision=True)
print(f"\n  TEST — LLaVA-Med-7B")
print(f"  Balanced Accuracy : {lv_bal:.4f}  |  Macro F1: {lv_f1:.4f}")
print(classification_report(lv_y_true, lv_y_pred, target_names=CLASS_NAMES, digits=4))
plot_confusion_matrix(lv_y_true, lv_y_pred, "LLaVA-Med-7B",
                      LLAVA_RESULTS_DIR, cmap="Reds")
run_stratified_analysis(lv_y_true, lv_y_pred, TEST_CSV,
                        "LLaVA-Med-7B", LLAVA_RESULTS_DIR)

## ⚪  SECTION 10 — MODEL 6: Qwen3-VL-8B

**What is Qwen3-VL?**

Qwen3-VL is the newest and most powerful model in this benchmark, released in 2025. It is an 8-billion parameter general Vision-Language Model with significantly improved reasoning abilities over its predecessor (Qwen2.5-VL), using a "thinking mode" architecture that allows it to reason through complex visual problems step by step.

**Why include it?**

It tests whether the newest generation of general AI can close the gap with medically-specialised models. Recent benchmarks show Qwen3-VL outperforms Qwen2.5-VL by ~18% on dermatology reasoning tasks — if that advantage holds on BCN20000, it has important implications for the field's direction.

### Cell 10.1 — Qwen3-VL settings

In [ ]:
# ⚙️ SETTINGS ─────────────────────────────────────────────────────────────────
QWEN3_MODEL_ID    = "Qwen/Qwen3-VL-8B-Instruct"  # 8B parameters
QWEN3_BATCH_SIZE  = 2     # largest model — use 2 on T4, 8 on A100
QWEN3_EPOCHS      = 5
QWEN3_LR          = 2e-4
QWEN3_GRAD_ACCUM  = 16    # effective batch = 2 * 16 = 32 on T4
QWEN3_PATIENCE    = 3
QWEN3_LORA_RANK   = 16
QWEN3_LORA_ALPHA  = 32
QWEN3_CHECKPOINT  = f"{CHECKPOINT_DIR}/qwen3_vl_8b_best"
QWEN3_RESULTS_DIR = f"{CHECKPOINT_DIR}/qwen3_vl_8b"
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(QWEN3_RESULTS_DIR, exist_ok=True)

### Cell 10.2 — Load Qwen3-VL model

⏱️ Download: ~16 GB, approximately 20–25 minutes on first run.

In [ ]:
qwen3_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
qwen3_lora_config = LoraConfig(
    r=QWEN3_LORA_RANK, lora_alpha=QWEN3_LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "up_proj", "down_proj", "gate_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)

qwen3_processor = AutoProcessor.from_pretrained(QWEN3_MODEL_ID)
# Qwen3-VL uses the same architecture class as Qwen2.5-VL
qwen3_base      = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN3_MODEL_ID,
    quantization_config=qwen3_bnb_config,
    device_map="auto", torch_dtype=torch.bfloat16,
)
qwen3_base  = prepare_model_for_kbit_training(qwen3_base)
qwen3_model = get_peft_model(qwen3_base, qwen3_lora_config)
qwen3_model.print_trainable_parameters()

### Cell 10.3 — Qwen3-VL DataLoaders and training

⏱️ Approximately 280 minutes on T4 / 80 minutes on A100 (5 epochs)

In [ ]:
qwen3_train_ds = BCN20000Dataset(TRAIN_CSV, processor=None, mode="vlm", augment=True)
qwen3_val_ds   = BCN20000Dataset(VAL_CSV,   processor=None, mode="vlm", augment=False)
qwen3_test_ds  = BCN20000Dataset(TEST_CSV,  processor=None, mode="vlm", augment=False)

qwen3_train_loader = DataLoader(qwen3_train_ds, batch_size=QWEN3_BATCH_SIZE,
                                 shuffle=True,  collate_fn=vlm_collate_fn)
qwen3_val_loader   = DataLoader(qwen3_val_ds,   batch_size=QWEN3_BATCH_SIZE,
                                 shuffle=False, collate_fn=vlm_collate_fn)
qwen3_test_loader  = DataLoader(qwen3_test_ds,  batch_size=QWEN3_BATCH_SIZE,
                                 shuffle=False, collate_fn=vlm_collate_fn)

qwen3_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, qwen3_model.parameters()),
    lr=QWEN3_LR, weight_decay=0.01,
)
qwen3_scaler = GradScaler("cuda")
best_q3_acc  = 0.0

print(f"\n{'═'*60}\n  Qwen3-VL-8B — BCN20000\n{'═'*60}\n")

for epoch in range(1, QWEN3_EPOCHS + 1):
    print(f"  Epoch {epoch}/{QWEN3_EPOCHS}")
    train_epoch_vlm(qwen3_model, qwen3_processor, qwen3_train_loader,
                    qwen3_optimizer, qwen3_scaler,
                    BCN20000Dataset.VLM_SYSTEM_PROMPT,
                    QWEN3_GRAD_ACCUM, mixed_precision=True)
    _, vl_bal, vl_f1, _, _, _ = eval_epoch_vlm(
        qwen3_model, qwen3_processor, qwen3_val_loader,
        BCN20000Dataset.VLM_SYSTEM_PROMPT, mixed_precision=True)
    print(f"  Val Balanced Acc: {vl_bal:.4f}  |  Val Macro F1: {vl_f1:.4f}")
    if vl_bal > best_q3_acc:
        best_q3_acc = vl_bal
        qwen3_model.save_pretrained(QWEN3_CHECKPOINT)
        print(f"  ★ New best: {best_q3_acc:.4f} — saved")

### Cell 10.4 — Evaluate Qwen3-VL on test set

In [ ]:
_, q3_bal, q3_f1, q3_y_true, q3_y_pred, _ = eval_epoch_vlm(
    qwen3_model, qwen3_processor, qwen3_test_loader,
    BCN20000Dataset.VLM_SYSTEM_PROMPT, mixed_precision=True)
print(f"\n  TEST — Qwen3-VL-8B")
print(f"  Balanced Accuracy : {q3_bal:.4f}  |  Macro F1: {q3_f1:.4f}")
print(classification_report(q3_y_true, q3_y_pred, target_names=CLASS_NAMES, digits=4))
plot_confusion_matrix(q3_y_true, q3_y_pred, "Qwen3-VL-8B",
                      QWEN3_RESULTS_DIR, cmap="Greys")
run_stratified_analysis(q3_y_true, q3_y_pred, TEST_CSV,
                        "Qwen3-VL-8B", QWEN3_RESULTS_DIR)

## 🏆  SECTION 11 — FINAL RESULTS COMPARISON

Run this section after ALL 6 models have been trained and evaluated. Produces the tables and figures for the paper's Results section.

### HOW TO INTERPRET THESE RESULTS

Balanced Accuracy is the primary metric — it is the average of each class's sensitivity (recall). A model that predicts "mole" for every image would get ~12.5% balanced accuracy (1/8 classes), even if its raw accuracy appears high.

Macro F1 captures the same idea from a precision-recall perspective. Macro AUROC measures how well each model separates each class from the rest.

For the paper, the most clinically important finding will be per-class sensitivity on Melanoma (MEL) — catching melanoma is the whole point.

### Cell 11.1 — Load all saved results

In [ ]:
MODEL_RESULT_FILES = {
    "DINOv2-Large"    : f"{CHECKPOINT_DIR}/dinov2_large/dinov2-large_results.json",
    "ConvNeXt V2-L"   : f"{CHECKPOINT_DIR}/convnextv2_large/convnext v2-large_results.json",
    "MedGemma-4B"     : f"{CHECKPOINT_DIR}/medgemma_4b/medgemma-4b_results.json",
    "Qwen2.5-VL-7B"   : f"{CHECKPOINT_DIR}/qwen25_vl_7b/qwen2.5-vl-7b_results.json",
    "LLaVA-Med-7B"    : f"{CHECKPOINT_DIR}/llava_med_7b/llava-med-7b_results.json",
    "Qwen3-VL-8B"     : f"{CHECKPOINT_DIR}/qwen3_vl_8b/qwen3-vl-8b_results.json",
}

all_results = []
for model_name, path in MODEL_RESULT_FILES.items():
    try:
        with open(path) as f:
            r = json.load(f)
            r["model"] = model_name
            all_results.append(r)
    except FileNotFoundError:
        print(f"  ⚠️  Results not found for {model_name} — run that model section first")

results_df = pd.DataFrame(all_results).sort_values("balanced_acc", ascending=False)

### Cell 11.2 — Print comparison table

In [ ]:
print(f"\n{'═'*70}")
print(f"  BENCHMARK RESULTS — BCN20000 Dermoscopy Classification")
print(f"  Ranked by Balanced Accuracy (primary metric)")
print(f"{'═'*70}")
print(f"  {'Rank':<6} {'Model':<22} {'Bal.Acc':<12} {'Macro F1':<12} {'AUROC':<10} {'Epoch'}")
print(f"  {'─'*65}")

for rank, (_, row) in enumerate(results_df.iterrows(), 1):
    winner = " ← BEST" if rank == 1 else ""
    print(f"  {rank:<6} {row['model']:<22} {row['balanced_acc']:<12.4f} "
          f"{row['macro_f1']:<12.4f} {row.get('macro_auroc', 'N/A'):<10} "
          f"{row.get('best_epoch', 'N/A')}{winner}")

results_df.to_csv(f"{CHECKPOINT_DIR}/final_comparison_table.csv", index=False)
print(f"\n  ✓ Table saved: {CHECKPOINT_DIR}/final_comparison_table.csv")

### Cell 11.3 — Bar chart comparison

In [ ]:
metrics  = ["balanced_acc", "macro_f1", "macro_auroc"]
labels   = ["Balanced Accuracy", "Macro F1", "Macro AUROC"]
x        = np.arange(len(results_df))
width    = 0.25
colors   = ["#3498db", "#2ecc71", "#e67e22"]

fig, ax = plt.subplots(figsize=(13, 6))
for i, (metric, label, color) in enumerate(zip(metrics, labels, colors)):
    vals = results_df[metric].fillna(0).values
    bars = ax.bar(x + i * width, vals, width, label=label, color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(results_df["model"].values, rotation=20, ha="right", fontsize=10)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title("Model Comparison — BCN20000 Dermoscopy Benchmark", fontsize=13)
ax.legend(fontsize=10)
ax.axhline(0.5, color="grey", linestyle=":", alpha=0.5, label="Chance (0.5)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/final_comparison_bars.png", dpi=150)
plt.show()
print(f"  ✓ Bar chart saved: {CHECKPOINT_DIR}/final_comparison_bars.png")

### Cell 11.4 — Per-class sensitivity heatmap

This is the key figure for the paper — shows which model catches which diagnosis best. Melanoma (MEL) sensitivity is the most critical column.

In [ ]:
sens_cols = [f"sens_{c}" for c in CLASS_NAMES]
available = [c for c in sens_cols if c in results_df.columns]

if available:
    sens_df = results_df.set_index("model")[available]
    sens_df.columns = CLASS_NAMES[:len(available)]

    fig, ax = plt.subplots(figsize=(13, len(results_df) * 0.9 + 2))
    sns.heatmap(
        sens_df.astype(float), annot=True, fmt=".2f",
        cmap="RdYlGn", vmin=0, vmax=1,
        linewidths=0.5, ax=ax,
        cbar_kws={"label": "Sensitivity (Recall)"},
    )
    ax.set_title(
        "Per-Class Sensitivity by Model\n"
        "(green = model catches this diagnosis well; red = frequently missed)",
        fontsize=12,
    )
    ax.set_xlabel("Diagnostic Category", fontsize=11)
    ax.set_ylabel("Model",               fontsize=11)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"{CHECKPOINT_DIR}/final_sensitivity_heatmap.png", dpi=150)
    plt.show()
    print(f"  ✓ Sensitivity heatmap saved: {CHECKPOINT_DIR}/final_sensitivity_heatmap.png")
else:
    print("  ℹ️  Per-class sensitivity not available — run full evaluation first")

### Cell 11.5 — Results paragraph template for the paper

Copy this into the Results section and fill in the bracketed placeholders.

In [ ]:
best_model = results_df.iloc[0]["model"]
best_bal   = results_df.iloc[0]["balanced_acc"]
best_f1    = results_df.iloc[0]["macro_f1"]

print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  PAPER RESULTS PARAGRAPH TEMPLATE                                           ║
║  Copy and edit this into your manuscript's Results section.                 ║
╚══════════════════════════════════════════════════════════════════════════════╝

We benchmarked six open-source AI models on the BCN20000 dermoscopy dataset,
evaluating performance using balanced accuracy as the primary metric, consistent
with the scoring protocol of the ISIC 2019 Challenge. Across [N] test images
spanning eight diagnostic categories, [BEST_MODEL] achieved the highest balanced
accuracy of [BEST_BAL_ACC] (macro-averaged F1 = [BEST_F1]; macro-AUROC =
[BEST_AUROC]), followed by [SECOND_MODEL] (balanced accuracy = [SECOND_BAL_ACC]).

Vision models demonstrated [superior/comparable] performance relative to
vision-language models: DINOv2-Large achieved a balanced accuracy of
[DINOV2_BAL_ACC] and ConvNeXt V2-Large achieved [CONVNEXT_BAL_ACC], compared
with [MEDGEMMA_BAL_ACC] for MedGemma-4B and [QWEN_BAL_ACC] for Qwen2.5-VL-7B.

Stratified analysis revealed that all models performed significantly better on
malignant than benign lesions (mean balanced accuracy [MAL_ACC] vs [BEN_ACC],
p < [P_VALUE]), consistent with the higher representation of malignant cases in
the training data and the more visually distinctive features of overtly malignant
lesions. Melanoma sensitivity varied substantially across models, ranging from
[MEL_SENS_LOW] to [MEL_SENS_HIGH], with [BEST_MEL_MODEL] demonstrating the
highest sensitivity for melanoma detection ([MEL_SENS]). Rare diagnostic
categories — including Dermatofibroma (DF) and Vascular Lesion (VASC) — were
consistently associated with the lowest per-class sensitivity across all models,
reflecting their underrepresentation in the training set (n = [DF_N] and n =
[VASC_N] respectively).

Current best model  : {best_model}
Balanced accuracy   : {best_bal:.4f}
Macro F1            : {best_f1:.4f}
""")

print("=" * 70)
print("  All notebook outputs saved to:", CHECKPOINT_DIR)
print("  Files produced:")
print("    class_distribution.png")
print("    dinov2_large/          — curves, confusion matrix, report, stratified")
print("    convnextv2_large/      — curves, confusion matrix, report, stratified")
print("    medgemma_4b/           — confusion matrix, report, stratified")
print("    qwen25_vl_7b/          — confusion matrix, report, stratified")
print("    llava_med_7b/          — confusion matrix, report, stratified")
print("    qwen3_vl_8b/           — confusion matrix, report, stratified")
print("    final_comparison_table.csv")
print("    final_comparison_bars.png")
print("    final_sensitivity_heatmap.png")
print("=" * 70)